# Repository RAG Research Playbook

This notebook walks through the same `uv`-managed repository RAG flow exposed by `make` and `uv run repo-rag`, with assertions and logging kept in tested Python helpers.


## 1. Resolve The Repository Root

### 1.1 Reuse The Research Playbook Scaffold

Keep notebook path resolution, smoke-test parsing, and baseline-answer setup in tested Python modules so the notebook stays thin.


In [1]:
from pathlib import Path

from repo_rag_lab.notebook_scaffolding import build_research_playbook_context
from repo_rag_lab.notebook_support import (
    assert_contains_text,
    resolve_repo_root,
    write_notebook_run_log,
)

root = resolve_repo_root(Path.cwd().resolve())
context = build_research_playbook_context(root)
print(root)


INFO repo_rag_lab.notebook_scaffolding Built research playbook context with 1 MCP candidates and smoke test manifest artifacts/azure/repo-rag-smoke.json.


/home/standard/dspy_rag_in_repo_docs_and_impl1


## 2. Confirm Shared Utility Surfaces

### 2.1 Start With The Same Commands Used By Agents And CI

Use `make utility-summary` or `uv run repo-rag utility-summary` before ad hoc experiments.


In [2]:
print(context["utility_summary"])


Repository utility surfaces:
- make utility-summary / uv run repo-rag utility-summary: list the supported entrypoints
- make ask / uv run repo-rag ask: answer repository-grounded questions
- make discover-mcp / uv run repo-rag discover-mcp: inspect repo-local MCP candidates
- make azure-manifest / uv run repo-rag azure-manifest: write Azure deployment metadata
- make smoke-test / uv run repo-rag smoke-test: validate the core workflow surfaces
- make verify-surfaces / uv run repo-rag verify-surfaces: validate notebooks and Makefile verification surfaces
- root: /home/standard/dspy_rag_in_repo_docs_and_impl1


## 3. Inspect MCP Server Candidates

### 3.1 Inventory Repo-Local MCP Hints

This shows the MCP discovery surface that can eventually replace parts of the baseline retriever.


In [3]:
context["mcp_candidates"]


['pyproject.toml']

## 4. Run Baseline Repository RAG

### 4.1 Ask A Question Grounded In Repository Files

Use a known repository question first so notebook output stays comparable with tests and smoke checks.


In [4]:
print(context["baseline_answer"])


Question: What does this repository research?

Evidence:
- /home/standard/dspy_rag_in_repo_docs_and_impl1/data/questions/repository.yaml: - question: What does this repository research? answer_contains: repository - question: Where are inspired implementation summaries stored? answer_contains: documentation/inspired - question: How should agents inspect repository utilities? 
- /home/standard/dspy_rag_in_repo_docs_and_impl1/tests/features/repository_rag.feature: Feature: Repository RAG Scenario: Answer a repository scope question Given the repository root When I ask the repository question "What does this repository research?" Then the answer mentions "repository" Scenario: Discover MCP candidates 
- /home/standard/dspy_rag_in_repo_docs_and_impl1/tests/test_benchmarks_and_notebook_scaffolding.py: oad = build_research_playbook_context(REPO_ROOT) assert "ask" in payload["utility_summary"] assert payload["baseline_question"] == "What does this repository research?" assert "Question:" in p

## 5. Run Smoke-Test Assertions And Log The Run

### 5.1 Confirm The Shared Utility Surfaces Still Agree

The playbook should assert utility-summary content, baseline-answer structure, smoke-test health, and then persist a notebook log.


In [5]:
assert_contains_text(
    context["utility_summary"],
    ["ask", "discover-mcp", "smoke-test", "verify-surfaces"],
    context="utility summary",
)
assert_contains_text(
    context["baseline_answer"],
    ["Question:", "Answer:", "Evidence:"],
    context="baseline answer",
)
smoke_test = context["smoke_test"]
assert smoke_test["answer_contains_repository"] is True
assert smoke_test["mcp_candidate_count"] == len(context["mcp_candidates"])
log_path = write_notebook_run_log(root, "01-repo-rag-research", context)
{
    "smoke_test": smoke_test,
    "log_path": str(log_path.relative_to(root)),
}


{'smoke_test': {'answer_contains_repository': True,
  'mcp_candidate_count': 1,
  'manifest_path': 'artifacts/azure/repo-rag-smoke.json'},
 'log_path': 'artifacts/notebook_logs/20260318T005557Z-01-repo-rag-research.json'}